# Find the users B lost

*Data Analysis in Python · Topic 9.3*

The checkout test failed its split check: B has 3,358 fewer users than A. Find the browsers where B lost its users, exclude them from both variants, and check the split again.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that reads the test's two files, the cell that builds the `users` table (topic 9.1.'s steps) and the cell with `srm_p_value()` (topic 9.2.'s function).
2. In the first cell marked ✏️, create the `browser_split` table: one row per `browser`, with the number of users in `A` and in `B` as columns, and a `share_b` column with B's share of the browser's users.
3. In the second cell marked ✏️, create `broken_browsers`: a list of the browsers where B has no users at all.
4. In the last cell marked ✏️, create:
   - the `fixed_users` table: the `users` table without the users of `broken_browsers`, in both variants
   - `fixed_srm_p`: `srm_p_value()`'s p-value for the `fixed_users` table's split
   - `fixed_lift`: B's conversion rate lift over A in the `fixed_users` table, as a fraction
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `browser_split` has 8 browsers, with B's share from 0% (`Android Webview` and `Safari (in-app)`) to 50.9% (`Safari`). `broken_browsers` is `["Android Webview", "Safari (in-app)"]`. The `fixed_users` table has 76,209 rows, `fixed_srm_p` is about `0.0777`, and `fixed_lift` about `0.0358`.

## Parameters

In [ ]:
import pandas as pd
from scipy import stats

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"

In [ ]:
# The test's two files, with the user IDs as text. Run it, and leave it as it is.
assignments = pd.read_csv(DATA_URL + "ab_test_assignments.csv", dtype={"user_pseudo_id": str}, parse_dates=["assigned_at"])
conversions = pd.read_csv(DATA_URL + "ab_test_conversions.csv", dtype={"user_pseudo_id": str}, parse_dates=["converted_at"])
assignments.shape, conversions.shape

In [ ]:
# One row per user in the test, with converted and revenue (topic 9.1). Run it, and leave it as it is.
users = assignments.merge(conversions[["user_pseudo_id", "revenue"]], on="user_pseudo_id", how="left", validate="one_to_one")
users["converted"] = users["revenue"].notna()
users["revenue"] = users["revenue"].fillna(0)
users.shape

In [ ]:
# The SRM check (topic 9.2). Run it, and leave it as it is.
def srm_p_value(counts):
    """Return the p-value of a chi-square test that the counts were split evenly between the variants."""
    return stats.chisquare(counts).pvalue


srm_p_value(users["variant"].value_counts())

## By browser

In [ ]:
browser_split = pd.crosstab(users["browser"], users["variant"])
browser_split["share_b"] = browser_split["B"] / (browser_split["A"] + browser_split["B"])
browser_split.sort_values("share_b")

## The browsers B lost

In [ ]:
broken_browsers = list(browser_split[browser_split["B"] == 0].index)
broken_browsers

## Without them

In [ ]:
fixed_users = users[~users["browser"].isin(broken_browsers)]
fixed_srm_p = srm_p_value(fixed_users["variant"].value_counts())
fixed_rates = fixed_users.groupby("variant")["converted"].mean()
fixed_lift = fixed_rates["B"] / fixed_rates["A"] - 1
len(fixed_users), fixed_srm_p, fixed_lift

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def number(value):
        return pd.api.types.is_number(value) and not isinstance(value, (pd.DataFrame, pd.Series))

    split = found.get("browser_split")
    if not isinstance(split, pd.DataFrame) or not {"A", "B", "share_b"} <= set(split.columns):
        results.append((False, "browser_split should be a DataFrame with the columns A, B and share_b: pd.crosstab(users[\"browser\"], users[\"variant\"]), then the share column."))
    elif len(split) != 8 or "Safari (in-app)" not in split.index:
        results.append((False, f"browser_split should have one row per browser, 8, with the browsers as its index. It has {len(split)} rows."))
    elif split["share_b"].max() > 1:
        results.append((False, "browser_split's share_b is in percent. Keep it as a fraction: B / (A + B)."))
    elif abs(split.loc["Safari", "share_b"] - 0.490619) < 0.000001:
        results.append((False, "browser_split's share_b is A's share: Safari's is 0.4906. Divide B by A + B."))
    elif split.loc["Chrome", "A"] != 25286 or abs(split.loc["Safari", "share_b"] - 0.509381) > 0.000001:
        results.append((False, f"browser_split should count each browser's users by variant: Chrome has 25,286 in A, and Safari's share_b is 0.5094. Yours has {split.loc['Chrome', 'A']:,} and {split.loc['Safari', 'share_b']:.4f}."))
    else:
        results.append((True, f"B's share of each browser's users runs from {split['share_b'].min():.1%} to {split['share_b'].max():.1%}."))

    broken = found.get("broken_browsers")
    if not isinstance(broken, (list, tuple, pd.Index, pd.Series)):
        results.append((False, f"broken_browsers should be a list of browser names: the browser_split table's rows where B is 0. It is {broken!r}."))
    elif set(broken) == {"Android Webview", "Safari (in-app)", "Firefox", "Other", "Samsung Internet"}:
        results.append((False, "broken_browsers has every browser where B's share is under 50%. A share of 47.8% is a coin's wobble; the brief asks for the browsers where B has no users at all."))
    elif set(broken) == {"Android Webview", "Safari (in-app)", "Safari"}:
        results.append((False, "broken_browsers has Safari, whose split fails at 0.05 (p = 0.014). With eight browsers checked, one failing at 0.05 is what chance does: Safari has 50.9% in B, not 0%. Keep the browsers where B has no users."))
    elif set(broken) != {"Android Webview", "Safari (in-app)"}:
        results.append((False, f"broken_browsers should be the two browsers where B has no users. It is {list(broken)}."))
    else:
        results.append((True, "B has no users in Android Webview or Safari (in-app): the in-app browsers of Instagram and Facebook, where the new checkout's users were never logged."))

    fixed = found.get("fixed_users")
    fixed_p, fixed_lift = found.get("fixed_srm_p"), found.get("fixed_lift")
    if not isinstance(fixed, pd.DataFrame) or "variant" not in fixed.columns:
        results.append((False, "fixed_users should be the users table without the broken browsers: users[~users[\"browser\"].isin(broken_browsers)]."))
    elif len(fixed) == 32100:
        results.append((False, "The fixed_users table has 32,100 rows: every mobile user is gone. That leaves out 44,109 mobile users whose split was fine. Exclude the two browsers, not the device."))
    elif len(fixed) in (77924, 78339):
        kept = "Android Webview" if len(fixed) == 77924 else "Safari (in-app)"
        results.append((False, f"The fixed_users table still has {kept}'s users, so its split still fails. Exclude both browsers."))
    elif len(fixed) != 76209:
        results.append((False, f"The fixed_users table should have the 76,209 users outside the two in-app browsers. It has {len(fixed):,}."))
    elif not number(fixed_p):
        results.append((False, f"fixed_srm_p should be a number: srm_p_value(fixed_users[\"variant\"].value_counts()). It is {fixed_p!r}."))
    elif abs(fixed_p - 0.077713) > 0.000001:
        results.append((False, f"fixed_srm_p should be the SRM check on the fixed_users table's split, 37,861 and 38,348: about 0.0777. It is {fixed_p:.4g}."))
    elif not number(fixed_lift):
        results.append((False, f"fixed_lift should be a number: B's conversion rate in the fixed_users table divided by A's, minus 1. It is {fixed_lift!r}."))
    elif abs(fixed_lift - 0.108668) < 0.000001:
        results.append((False, "fixed_lift is 0.1087: that's the lift of all users. Calculate it from the fixed_users table."))
    elif abs(fixed_lift - 0.035786) > 0.000001:
        results.append((False, f"fixed_lift should be B's conversion rate divided by A's, minus 1, in the fixed_users table: about 0.0358. It is {fixed_lift:.4f}."))
    else:
        results.append((True, f"Without the in-app browsers, the split holds (p = {fixed_p:.3f}, above 0.001), and B's lift drops from +10.9% to {fixed_lift:+.1%}. In-app users convert worse, and only A had them."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
browser_split = pd.crosstab(users["browser"], users["variant"])
browser_split["share_b"] = browser_split["B"] / (browser_split["A"] + browser_split["B"])
browser_split.sort_values("share_b")
```

```python
broken_browsers = list(browser_split[browser_split["B"] == 0].index)
broken_browsers
```

```python
fixed_users = users[~users["browser"].isin(broken_browsers)]
fixed_srm_p = srm_p_value(fixed_users["variant"].value_counts())
fixed_rates = fixed_users.groupby("variant")["converted"].mean()
fixed_lift = fixed_rates["B"] / fixed_rates["A"] - 1
len(fixed_users), fixed_srm_p, fixed_lift
```

</details>